# Les 04 - Tool Use Ontwerppatroon

In deze les leer je het **Tool Use** ontwerppatroon voor AI-agenten met behulp van het Microsoft Agent Framework (Python). We behandelen:

- Het definiëren van functietools met de `@tool` decorator en getypeerde parameters
- Het bieden van toolschema's zodat het model weet wat elke tool doet
- Het beheersen van tooluitvoering met `approval_mode`
- Het retourneren van **gestructureerde output** via Pydantic-modellen en `response_format`

Het scenario is een **reisboekingsagent** die bestemmingen kan opzoeken, beschikbaarheid kan controleren en vluchtinformatie kan ophalen.


## Installatie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Tools definiëren met de @tool Decorator

De `@tool` decorator verandert een gewone Python functie in een tool die een agent kan aanroepen.
Belangrijke punten:

- De **docstring** wordt de toolbeschrijving die het model ziet.
- **Type-aanduidingen** (inclusief `Annotated` met beschrijvingen) definiëren het toolschema.
- `approval_mode` bepaalt of de gebruiker elke aanroep moet goedkeuren voordat deze wordt uitgevoerd.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Een agent maken met meerdere tools

Geef alle drie de tools door aan de client zodat het model de tools kan aanroepen die het nodig heeft om de vraag van de gebruiker te beantwoorden.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Gestructureerde Uitvoer met Hulpmiddelen

Door `response_format` in te stellen op een Pydantic-model, wordt de agent gedwongen een goed getypeerd JSON-object terug te geven in plaats van vrije tekst. Dit is handig wanneer code die daarna komt het resultaat programmatisch moet verwerken.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Patronen voor goedkeuring van tools

De parameter `approval_mode` op `@tool` bepaalt of tool-aanroepen menselijke goedkeuring vereisen voordat ze worden uitgevoerd:

| Modus | Gedrag |
|---|---|
| `"never_require"` | Tool wordt automatisch uitgevoerd — geen gebruikersbevestiging nodig. |
| `"always_require"` | Elke aanroep moet door de gebruiker worden goedgekeurd voordat deze wordt uitgevoerd. |

Gebruik `"always_require"` voor tools die bijwerkingen hebben (bijv. het boeken van een vlucht, het in rekening brengen van een creditcard) zodat een mens betrokken blijft.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Samenvatting

In deze les heb je geleerd hoe je:

1. **Tools definieert** met behulp van de `@tool` decorator met getypte parameters en docstrings die dienen als het toolschema.
2. **Meerdere tools samenstelt** zodat de agent ze achtereenvolgens kan aanroepen om complexe vragen te beantwoorden.
3. **Gestructureerde uitvoer retourneert** door een Pydantic-model als `response_format` mee te geven.
4. **Toolgoedkeuring beheerst** met `approval_mode` om een mens in de lus te houden voor gevoelige bewerkingen.

Deze patronen vormen de basis voor het bouwen van betrouwbare, productieklare agenten die veilig kunnen communiceren met externe systemen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
